<a href="https://colab.research.google.com/github/sachin23-an/Quant-Finance-Projects/blob/main/Institutional_Liquidity_Heatmap_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Institutional Liquidity Heatmap Model: Project Overview

This notebook presents an advanced model designed to visualize market liquidity and its dynamic interaction with price movements. We will identify key liquidity zones, apply a sophisticated weighting system, and leverage a rolling Kernel Density Estimate (KDE) to generate interactive 2D heatmaps, immersive 3D surface plots, and a time-based slider for in-depth analysis.

### **Our Goal:**
To develop a robust model that clearly illustrates market liquidity, demonstrates how price reacts to these zones, and provides actionable insights and timely alerts for strategic decision-making.

## Setup: Installing Dependencies and Importing Libraries

Before we begin, let's install the necessary Python libraries. We'll be using `yfinance` to fetch historical data, `plotly` for our interactive visualizations, `scipy` for statistical calculations like Kernel Density Estimation, and `numpy` along with `pandas` for efficient data manipulation. We'll also filter out any warnings to keep our output clean.

In [ ]:
!pip install yfinance plotly scipy numpy pandas --quiet

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import yfinance as yf
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy.stats import gaussian_kde
from scipy.ndimage import gaussian_filter
from datetime import datetime, timedelta

## User Configuration

Edit these values to customize the data you wish to analyze. The defaults are now set for Nifty 50 (`^NSEI`) over 60 days with daily intervals, providing an Indian market perspective.

In [ ]:
TICKER     = "^NSEI"   # Nifty 50 index for Indian market perspective. e.g. "AAPL", "ETH-USD", "SPY"
TIMEFRAME  = 180         # days of history (increased for more data points for animation)
INTERVAL   = "1d"        # "1h", "30m", "1d" (changed to 1d for better Nifty 50 data availability)

## Step 1: Data Download

This step fetches historical OHLCV (Open, High, Low, Close, Volume) data for the specified ticker, timeframe, and interval using `yfinance`. The data is cleaned by dropping any rows with missing values, and multi-level columns (if any) are flattened.

In [ ]:
def fetch_data(ticker: str, timeframe: int, interval: str) -> pd.DataFrame:
    """Download OHLCV data via yfinance and clean it."""
    end   = datetime.now()
    start = end - timedelta(days=timeframe)
    print(f"[DATA] Downloading {ticker} | {interval} | last {timeframe} days …")
    df = yf.download(ticker, start=start, end=end, interval=interval, progress=False)
    df.dropna(inplace=True)
    df.columns = [c[0] if isinstance(c, tuple) else c for c in df.columns]
    print(f"[DATA] {len(df)} candles loaded  ({df.index[0]} → {df.index[-1]})")
    return df

## Step 2: Liquidity Detection

This step identifies various types of liquidity levels from the historical price data. It includes functions to detect:

*   **Swing Highs:** Local price maxima within a defined rolling window.
*   **Swing Lows:** Local price minima within a defined rolling window.
*   **High Volume Zones:** Price levels where trading volume exceeds a rolling average, indicating significant interest.
*   **Psychological Levels:** Rounded price levels that often act as support or resistance due to human psychology.

All detected liquidity levels are then collected into a single list for further processing.

In [ ]:
def detect_swing_highs(df: pd.DataFrame, window: int = 10) -> np.ndarray:
    highs = df["High"].values
    levels = []
    for i in range(window, len(highs) - window):
        if highs[i] == max(highs[i - window: i + window + 1]):
            levels.append((i, highs[i], "swing_high"))
    return levels


def detect_swing_lows(df: pd.DataFrame, window: int = 10) -> np.ndarray:
    lows = df["Low"].values
    levels = []
    for i in range(window, len(lows) - window):
        if lows[i] == min(lows[i - window: i + window + 1]):
            levels.append((i, lows[i], "swing_low"))
    return levels


def detect_high_volume_zones(df: pd.DataFrame, window: int = 20) -> list:
    vol_mean = df["Volume"].rolling(window).mean()
    hv_idx   = df.index[df["Volume"] > vol_mean].tolist()
    levels   = []
    for idx in hv_idx:
        pos = df.index.get_loc(idx)
        price = (df["High"].iloc[pos] + df["Low"].iloc[pos]) / 2
        levels.append((pos, price, "high_volume"))
    return levels


def detect_psychological_levels(df: pd.DataFrame) -> list:
    price_min = df["Low"].min()
    price_max = df["High"].max()
    price_range = price_max - price_min

    # choose rounding granularity based on price magnitude
    if price_max > 10_000:
        step = 1000
    elif price_max > 1_000:
        step = 500
    elif price_max > 100:
        step = 100
    elif price_max > 10:
        step = 10
    else:
        step = 1

    levels_100 = np.arange(
        np.floor(price_min / step) * step,
        np.ceil(price_max  / step) * step + step,
        step
    )
    levels_500 = np.arange(
        np.floor(price_min / (step * 5)) * (step * 5),
        np.ceil(price_max  / (step * 5)) * (step * 5) + step * 5,
        step * 5
    )
    all_psych = np.unique(np.concatenate([levels_100, levels_500]))
    result = []
    for p in all_psych:
        if price_min <= p <= price_max:
            result.append((-1, p, "psychological"))   # -1 → no specific time index
    return result


def collect_liquidity_levels(df: pd.DataFrame) -> list:
    """Merge all liquidity sources into one list of (time_idx, price, type)."""
    levels  = []
    levels += detect_swing_highs(df)
    levels += detect_swing_lows(df)
    levels += detect_high_volume_zones(df)
    levels += detect_psychological_levels(df)
    print(f"[LIQUIDITY] {len(levels)} raw levels detected")
    return levels

## Step 3: Weighting System

To better reflect the market's perception of liquidity, a weighting system is applied. This system prioritizes:

*   **Volume-based levels:** Higher weight for areas of significant trading activity.
*   **Recent data:** More recent liquidity levels are considered more relevant, incorporating a time decay factor.
*   **Swing points:** Significant turning points in price are assigned a medium weight.

Psychological levels receive a base weight.

In [ ]:
def compute_weights(levels: list, n_bars: int) -> tuple[np.ndarray, np.ndarray]:
    """Return (prices_array, weights_array) for all liquidity events."""
    type_weight = {
        "high_volume"   : 3.0,
        "swing_high"    : 2.0,
        "swing_low"     : 2.0,
        "psychological" : 1.5,
    }
    prices  = []
    weights = []
    for (idx, price, ltype) in levels:
        base_w = type_weight.get(ltype, 1.0)
        # time decay: recent → higher weight
        if idx >= 0:
            recency = idx / max(n_bars - 1, 1)   # 0 (old) → 1 (new)
            time_w  = 0.5 + 0.5 * recency
        else:
            time_w = 0.75                          # psych levels get mid weight
        prices.append(price)
        weights.append(base_w * time_w)
    return np.array(prices), np.array(weights)

## Step 4: Rolling Kernel Density Estimate (KDE)

This core engine transforms discrete liquidity levels into a continuous density representation using a **rolling window approach** to capture the evolution of liquidity over time. For each time step:

1.  **Relevant liquidity levels** within the current rolling window are selected.
2.  A **Gaussian KDE** is applied to these levels, incorporating the computed weights.
3.  The resulting density values are **normalized** between 0 and 1.

This process generates a `Z` matrix, where `X` is time, `Y` is the price grid, and `Z` is the normalized liquidity density, providing a dynamic view of market liquidity.

In [ ]:
def build_rolling_kde(
    df: pd.DataFrame,
    levels: list,
    price_grid_pts: int = 200,
    kde_window: int = 50
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Returns:
        time_axis  : shape (T,)
        price_grid : shape (P,)
        Z          : shape (P, T)  — normalized liquidity density
    """
    n_bars     = len(df)

    # Ensure enough data is available for KDE calculation
    if n_bars < 2:
        print(f"[KDE Error] Not enough data ({n_bars} bars) to build rolling KDE. Minimum 2 bars required.")
        return np.array([]), np.array([]), np.array([]) # Return empty arrays

    # Adjust kde_window if it's too large for the available data
    original_kde_window = kde_window
    if kde_window >= n_bars:
        kde_window = max(1, n_bars - 1) # Ensure kde_window is at least 1
        print(f"[KDE Warning] kde_window ({original_kde_window}) was too large for {n_bars} bars. Adjusted to {kde_window}.")
    elif kde_window == 0: # Ensure kde_window is not 0
        kde_window = 1
        print(f"[KDE Warning] kde_window was 0. Adjusted to {kde_window}.")

    close      = df["Close"].values
    price_min  = df["Low"].min()  * 0.995
    price_max  = df["High"].max() * 1.005
    price_grid = np.linspace(price_min, price_max, price_grid_pts)

    # Pre-compute weights for all levels
    all_prices, all_weights = compute_weights(levels, n_bars)

    T          = n_bars - kde_window
    Z          = np.zeros((price_grid_pts, T))
    time_axis  = np.arange(kde_window, n_bars)

    print(f"[KDE] Building rolling KDE over {T} time steps …")
    for t in range(T):
        window_start = t
        window_end   = t + kde_window
        close_win    = close[window_start:window_end]
        c_min, c_max = close_win.min(), close_win.max()

        # Filter levels that are "visible" inside this price window (±20%)
        margin = (c_max - c_min) * 0.5
        mask   = (all_prices >= c_min - margin) & (all_prices <= c_max + margin)
        w_prices  = all_prices[mask]
        w_weights = all_weights[mask]

        if len(w_prices) < 3:
            continue

        # Also inject close prices (current bar region) as background density
        bg_prices  = close_win
        bg_weights = np.ones(len(bg_prices)) * 0.3

        combined_p = np.concatenate([w_prices, bg_prices])
        combined_w = np.concatenate([w_weights, bg_weights])

        try:
            kde = gaussian_kde(combined_p, weights=combined_w, bw_method=0.15)
            density = kde(price_grid)
            # Normalize
            dmax = density.max()
            if dmax > 0:
                density /= dmax
            Z[:, t] = density
        except Exception:
            pass

    # Smooth Z lightly in time dimension to remove artifacts
    Z = gaussian_filter(Z, sigma=[1.5, 1.0])
    print("[KDE] Done.")
    return time_axis, price_grid, Z

## Step 5: 2D Interactive Heatmap

The primary output is an interactive 2D heatmap that visualizes liquidity density over time and price. This plot features:

*   **X-axis:** Time (dates).
*   **Y-axis:** Price levels.
*   **Color:** Represents liquidity density (Z-matrix values).
*   **Price Overlay:** The actual closing price is overlaid as a bright cyan line to show its interaction with liquidity zones.

`Plotly` is used to ensure interactivity, allowing users to zoom, pan, and inspect data points.

In [ ]:
def plot_2d_heatmap(
    df: pd.DataFrame,
    time_axis: np.ndarray,
    price_grid: np.ndarray,
    Z: np.ndarray
) -> go.Figure:
    """Interactive 2D heatmap with price overlay."""
    timestamps = df.index[time_axis].astype(str).tolist()
    close_line = df["Close"].values[time_axis]

    fig = go.Figure()

    # ── Heatmap layer
    fig.add_trace(go.Heatmap(
        z          = Z,
        x          = timestamps,
        y          = price_grid,
        colorscale = "Plasma", # Changed to a more vibrant built-in colorscale
        zsmooth    = "best",
        showscale  = True,
        colorbar   = dict(
            title      = "Liquidity<br>Density",
            tickfont   = dict(color="#aaa", size=11),
            titlefont  = dict(color="#ccc", size=12),
            bgcolor    = "rgba(0,0,0,0)",
            bordercolor= "rgba(255,255,255,0.1)",
        ),
        hovertemplate = "Time: %{x}<br>Price: %{y:.2f}<br>Density: %{z:.3f}<extra></extra>",
    ))

    # ── Price overlay
    fig.add_trace(go.Scatter(
        x    = timestamps,
        y    = close_line,
        mode = "lines",
        line = dict(color="cyan", width=1.8, dash="solid"),
        name = "Close Price",
        hovertemplate = "Price: %{y:.2f}<extra>Close</extra>",
    ))

    fig.update_layout(
        title      = dict(
            text      = f"<b>Institutional Liquidity Heatmap — {TICKER}</b>",
            font      = dict(size=20, color="#e0e0ff"),
            x         = 0.5,
        ),
        paper_bgcolor = "rgb(5,5,15)",
        plot_bgcolor  = "rgb(8,8,25)",
        font          = dict(color="#aaa"),
        xaxis = dict(
            title      = "Time",
            showgrid   = False,
            tickfont   = dict(size=10),
            rangeslider= dict(visible=True, bgcolor="rgb(15,15,35)", thickness=0.05),
        ),
        yaxis = dict(
            title    = f"Price ({TICKER})",
            showgrid = True,
            gridcolor= "rgba(255,255,255,0.04)",
            tickfont = dict(size=10),
        ),
        legend = dict(
            bgcolor    = "rgba(0,0,0,0.5)",
            bordercolor= "rgba(255,255,255,0.1)",
            font       = dict(color="#ccc"),
        ),
        height = 600,
        margin = dict(l=60, r=40, t=70, b=60),
    )
    return fig

## Step 6: 3D Surface Plot

To provide another perspective, a 3D surface plot visualizes liquidity density across time and price in three dimensions:

*   **X-axis:** Time.
*   **Y-axis:** Price.
*   **Z-axis:** Liquidity Density.

The plot features a smooth surface, a custom color scale, and lighting effects to enhance visual depth. The closing price line is also overlaid above the surface.

In [ ]:
def plot_3d_surface(
    df: pd.DataFrame,
    time_axis: np.ndarray,
    price_grid: np.ndarray,
    Z: np.ndarray
) -> go.Figure:
    """3D liquidity surface with price ribbon above."""
    timestamps  = df.index[time_axis].astype(str).tolist()
    close_line  = df["Close"].values[time_axis]
    t_numeric   = np.arange(len(time_axis))

    fig = go.Figure()

    # ── Surface
    fig.add_trace(go.Surface(
        z          = Z,
        x          = t_numeric,
        y          = price_grid,
        colorscale = [
            [0.0,  "rgb(0,0,10)"],
            [0.25, "rgb(60,0,100)"],
            [0.5,  "rgb(160,20,60)"],
            [0.75, "rgb(240,100,10)"],
            [1.0,  "rgb(255,240,50)"],
        ],
        opacity   = 0.92,
        showscale = True,
        colorbar  = dict(
            title    = "Liquidity<br>Density",
            tickfont = dict(color="#aaa"),
            titlefont= dict(color="#ccc"),
        ),
        lighting  = dict(
            ambient   = 0.4,
            diffuse   = 0.7,
            specular  = 0.4,
            roughness = 0.6,
            fresnel   = 0.3,
        ),
        contours  = dict(
            z=dict(show=True, usecolormap=True, highlightcolor="white", project_z=True)
        ),
    ))

    # ── Price ribbon above surface
    z_price = close_line.copy()
    fig.add_trace(go.Scatter3d(
        x    = t_numeric,
        y    = z_price,
        z    = np.ones_like(t_numeric) * 1.05,  # slightly above max density
        mode = "lines",
        line = dict(color="cyan", width=4),
        name = "Close Price",
    ))

    # Tick labels on x axis (every N steps)
    n_ticks   = 8
    tick_step = max(len(time_axis) // n_ticks, 1)
    tick_vals = list(range(0, len(time_axis), tick_step))
    tick_text = [timestamps[i] for i in tick_vals]

    fig.update_layout(
        title      = dict(
            text = f"<b>3D Liquidity Surface — {TICKER}</b>",
            font = dict(size=18, color="#e0e0ff"),
            x    = 0.5,
        ),
        paper_bgcolor = "rgb(5,5,15)",
        scene = dict(
            bgcolor = "rgb(5,5,20)",
            xaxis   = dict(
                title     = "Time",
                tickvals  = tick_vals,
                ticktext  = tick_text,
                tickfont  = dict(size=9, color="#888"),
                gridcolor = "rgba(255,255,255,0.05)",
                showbackground=False,
            ),
            yaxis   = dict(
                title     = "Price",
                tickfont  = dict(size=9, color="#888"),
                gridcolor = "rgba(255,255,255,0.05)",
                showbackground=False,
            ),
            zaxis   = dict(
                title     = "Density",
                tickfont  = dict(size=9, color="#888"),
                gridcolor = "rgba(255,255,255,0.05)",
                showbackground=False,
            ),
            camera  = dict(eye=dict(x=1.6, y=-1.8, z=0.8)),
        ),
        height = 650,
        margin = dict(l=0, r=0, t=60, b=0),
    )
    return fig

## Step 7: Slider-based Animation

This interactive visualization allows users to scroll through time and observe the liquidity distribution at each specific time step. A 2D density bar chart for the selected time slice is displayed, showing how liquidity changes dynamically along with the current price.

In [ ]:
def plot_slider_animation(
    df: pd.DataFrame,
    time_axis: np.ndarray,
    price_grid: np.ndarray,
    Z: np.ndarray,
    n_frames: int = 40
) -> go.Figure:
    """Animated slider — liquidity distribution at each time step."""
    close_vals = df["Close"].values[time_axis]
    timestamps = df.index[time_axis].astype(str).tolist()
    step_size  = max(len(time_axis) // n_frames, 1)
    frame_idx  = list(range(0, len(time_axis), step_size))

    frames = []
    for fi in frame_idx:
        density_col = Z[:, fi]
        current_price = close_vals[fi]
        frames.append(go.Frame(
            data=[
                go.Bar(
                    x         = density_col,
                    y         = price_grid,
                    orientation = "h",
                    marker    = dict(
                        color     = density_col,
                        colorscale= "Plasma", # Changed to Plasma for consistency
                        cmin = 0, cmax = 1,
                        showscale = False,
                    ),
                    name      = "Liquidity",
                    hovertemplate = "Price: %{y:.2f}<br>Density: %{x:.3f}<extra></extra>",
                ),
                go.Scatter(
                    x    = [0, density_col.max() * 1.05],
                    y    = [current_price, current_price],
                    mode = "lines",
                    line = dict(color="cyan", width=2, dash="dash"),
                    name = "Current Price",
                ),
            ],
            name   = str(fi),
            layout = go.Layout(
                title_text = f"<b>Liquidity Distribution — {timestamps[fi]}</b>"
            )
        ))

    # Initial frame
    init_density = Z[:, 0]
    init_price   = close_vals[0]

    fig = go.Figure(
        data=[
            go.Bar(
                x           = init_density,
                y           = price_grid,
                orientation = "h",
                marker      = dict(
                    color      = init_density,
                    colorscale = "Plasma", # Changed to Plasma for consistency
                    cmin=0, cmax=1, showscale=True,
                    colorbar=dict(title="Density", tickfont=dict(color="#aaa")),
                ),
                name = "Liquidity",
            ),
            go.Scatter(
                x    = [0, init_density.max() * 1.05],
                y    = [init_price, init_price],
                mode = "lines",
                line = dict(color="cyan", width=2, dash="dash"),
                name = "Current Price",
            ),
        ],
        layout=go.Layout(
            title      = dict(
                text = f"<b>Liquidity Distribution — {timestamps[0]}</b>",
                font = dict(size=17, color="#e0e0ff"),
                x     = 0.5,
            ),
            paper_bgcolor = "rgb(5,5,15)",
            plot_bgcolor  = "rgb(8,8,25)",
            font          = dict(color="#aaa"),
            xaxis = dict(title="Liquidity Density", showgrid=True, gridcolor="rgba(255,255,255,0.05)"),
            yaxis = dict(title=f"Price ({TICKER})", showgrid=True, gridcolor="rgba(255,255,255,0.05)"),
            height = 600,
            margin = dict(l=60, r=40, t=70, b=80),
            updatemenus=[dict(
                type       = "buttons",
                showactive = False,
                y=1.08, x=0.08,
                buttons=[
                    dict(label="▶ Play",
                         method="animate",
                         args=[None, dict(
                             frame        = dict(duration=120, redraw=True),
                             fromcurrent  = True,
                             transition   = dict(duration=60),
                         )]),
                    dict(label="⏸ Pause",
                         method="animate",
                         args=[[None], dict(
                             frame       = dict(duration=0, redraw=False),
                             mode        = "immediate",
                             transition  = dict(duration=0),
                         )]),
                ],
            )],
            sliders=[dict(
                active     = 0,
                steps      = [dict(
                    method = "animate",
                    args   = [[str(fi)], dict(
                        mode        = "immediate",
                        frame       = dict(duration=100, redraw=True),
                        transition  = dict(duration=50),
                    )],
                    label = timestamps[fi][:10],
                ) for fi in frame_idx],
                x=0, y=0,
                len=1.0,
                pad=dict(b=10, t=50),
                currentvalue=dict(
                    prefix    = "Time: ",
                    visible   = True,
                    xanchor   = "center",
                    font      = dict(size=13, color="#ccc"),
                ),
                font=dict(size=9, color="#888"),
            )],
        ),
        frames=frames,
    )
    return fig

## Step 8: Interpretation Engine

This engine provides market insights by analyzing the current price against significant liquidity zones. It identifies the nearest strong liquidity areas above and below the current price and suggests potential price attraction based on the density of these zones. A strong liquidity zone acts as a magnet for price.

In [ ]:
def interpret_market(
    df: pd.DataFrame,
    price_grid: np.ndarray,
    Z: np.ndarray,
    time_axis: np.ndarray,
    top_n: int = 5
) -> None:
    """Print market insight based on current price vs liquidity zones."""
    current_price = float(df["Close"].iloc[-1])
    last_density  = Z[:, -1]

    # Find top liquidity peaks
    from scipy.signal import find_peaks
    peaks, props = find_peaks(last_density, height=0.3, distance=5)

    peak_prices = price_grid[peaks]
    peak_dens   = last_density[peaks]
    sorted_idx  = np.argsort(peak_dens)[::-1]
    peak_prices = peak_prices[sorted_idx]
    peak_dens   = peak_dens[sorted_idx]

    above = [(p, d) for p, d in zip(peak_prices, peak_dens) if p > current_price]
    below = [(p, d) for p, d in zip(peak_prices, peak_dens) if p < current_price]

    nearest_above = above[0]  if above else None
    nearest_below = below[-1] if below else None

    sep = "=" * 60
    print(f"\n{sep}")
    print(f"  MARKET INTERPRETATION ENGINE — {TICKER}")
    print(sep)
    print(f"  Current Price : {current_price:,.2f}")

    if nearest_above:
        dist_up  = (nearest_above[0] - current_price) / current_price * 100
        print(f"  Nearest Liq ↑ : {nearest_above[0]:,.2f}  (density={nearest_above[1]:.2f}, +{dist_up:.2f}%) fidelity)")
    if nearest_below:
        dist_dn  = (current_price - nearest_below[0]) / current_price * 100
        print(f"  Nearest Liq ↓ : {nearest_below[0]:,.2f}  (density={nearest_below[1]:.2f}, -{dist_dn:.2f}%) fidelity)")

    print(f"\n  Top {min(top_n, len(peak_prices))} Liquidity Zones:")
    for i, (p, d) in enumerate(zip(peak_prices[:top_n], peak_dens[:top_n])):
        direction = "↑ ABOVE" if p > current_price else "↓ BELOW"
        bar = "█" * int(d * 20)
        print(f"    {i+1}. {p:>12,.2f}  {direction}  [{bar:<20}] {d:.2f}")

    print()
    # Magnet logic
    if nearest_above and nearest_below:
        if nearest_above[1] > nearest_below[1] * 1.2:
            print(f"  ⚡ Market Insight: Price is likely attracted UPWARD toward {nearest_above[0]:,.2f}")
            print(f"     Strong liquidity magnet detected above current price.")
        elif nearest_below[1] > nearest_above[1] * 1.2:
            print(f"  ⚡ Market Insight: Price is likely pulled DOWNWARD toward {nearest_below[0]:,.2f}")
            print(f"     Strong liquidity pool detected below current price.")
        else:
            print(f"  ⚡ Market Insight: Price is in a BALANCED liquidity zone.")
            print(f"     Equal attraction above ({nearest_above[0]:,.2f}) and below ({nearest_below[0]:,.2f}).")
    elif nearest_above:
        print(f"  ⚡ Market Insight: Price is likely attracted toward {nearest_above[0]:,.2f} (only liquidity above)")
    elif nearest_below:
        print(f"  ⚡ Market Insight: Price is likely attracted toward {nearest_below[0]:,.2f} (only liquidity below)")
    print(sep + "\n")

## Step 9: Alert System

This system provides real-time alerts when the price approaches a high-density liquidity zone, indicating a potential area of strong interaction. An alert is triggered if:

*   The price is within a defined proximity percentage (e.g., 1%) of a high-density zone.
*   The liquidity density in that zone is above a specified threshold (e.g., 0.6 on a scale of 0 to 1).

This helps traders identify critical price levels quickly.

In [ ]:
def check_alerts(
    df: pd.DataFrame,
    price_grid: np.ndarray,
    Z: np.ndarray,
    proximity_pct: float = 0.01,
    density_threshold: float = 0.6
) -> None:
    """Alert when price is within 1% of a high-density zone."""
    current_price = float(df["Close"].iloc[-1])
    last_density  = Z[:, -1]

    alerts_fired = 0
    for i, (pg, dens) in enumerate(zip(price_grid, last_density)):
        if dens >= density_threshold:
            dist_pct = abs(pg - current_price) / current_price
            if dist_pct <= proximity_pct:
                alerts_fired += 1
                direction = "ABOVE" if pg > current_price else "BELOW"
                dist_str  = f"{dist_pct*100:.3f}%"
                print(f"  🔴 ALERT  | High Liquidity Interaction Zone Detected")
                print(f"         Price: {current_price:,.2f}  |  Zone: {pg:,.2f}  ({direction}, {dist_str} away)")
                print(f"         Density: {dens:.2f}  [threshold={density_threshold}]")
                print()

    if alerts_fired == 0:
        print("  ✅ No active liquidity alerts — price is not in a high-density interaction zone.\n")

## Main Pipeline

The `main` function orchestrates the entire process, calling each step sequentially to download data, detect liquidity, compute KDE, render visualizations, provide market interpretation, and check for alerts.

In [ ]:

def main():
    # 1. Data
    df = fetch_data(TICKER, TIMEFRAME, INTERVAL)

    # 2. Liquidity Detection
    levels = collect_liquidity_levels(df)

    # 3–4. Weights + Rolling KDE
    time_axis, price_grid, Z = build_rolling_kde(df, levels)

    # 5. 2D Heatmap
    print("[VIZ] Rendering 2D heatmap …")
    fig2d = plot_2d_heatmap(df, time_axis, price_grid, Z)
    fig2d.show()

    # 6. 3D Surface
    print("[VIZ] Rendering 3D surface …")
    fig3d = plot_3d_surface(df, time_axis, price_grid, Z)
    fig3d.show()

    # 7. Slider / Animation
    print("[VIZ] Rendering animated slider …")
    fig_slider = plot_slider_animation(df, time_axis, price_grid, Z)
    fig_slider.show()

    # 8. Interpretation Engine
    interpret_market(df, price_grid, Z, time_axis)

    # 9. Alert System
    print("=" * 60)
    print("  ALERT SYSTEM")
    print("=" * 60)
    check_alerts(df, price_grid, Z)

## Run the Model

Execute the main pipeline to generate the liquidity heatmap and analysis.

In [ ]:
main()

[DATA] Downloading ^NSEI | 1d | last 180 days …
[DATA] 120 candles loaded  (2025-10-28 00:00:00 → 2026-04-24 00:00:00)
[LIQUIDITY] 60 raw levels detected
[KDE] Building rolling KDE over 70 time steps …
[KDE] Done.
[VIZ] Rendering 2D heatmap …


[VIZ] Rendering 3D surface …


[VIZ] Rendering animated slider …



  MARKET INTERPRETATION ENGINE — ^NSEI
  Current Price : 23,897.95
  Nearest Liq ↑ : 25,970.38  (density=1.00, +8.67%) fidelity)
  Nearest Liq ↓ : 23,185.56  (density=0.58, -2.98%) fidelity)

  Top 4 Liquidity Zones:
    1.    25,970.38  ↑ ABOVE  [███████████████████ ] 1.00
    2.    25,413.42  ↑ ABOVE  [█████████████████   ] 0.86
    3.    23,185.56  ↓ BELOW  [███████████         ] 0.58
    4.    23,898.48  ↑ ABOVE  [███████             ] 0.36

  ⚡ Market Insight: Price is likely attracted UPWARD toward 25,970.38
     Strong liquidity magnet detected above current price.

  ALERT SYSTEM
  ✅ No active liquidity alerts — price is not in a high-density interaction zone.



## Conclusion

This project successfully developed an Institutional Liquidity Heatmap Model, providing a comprehensive framework for visualizing and interpreting market liquidity. By detecting various liquidity sources, applying a dynamic weighting system, and utilizing rolling Kernel Density Estimation, we've created powerful interactive visualizations including 2D heatmaps, 3D surface plots, and a time-slider animation.

The market interpretation engine and alert system further enhance this model by offering actionable insights and timely notifications, enabling traders and analysts to better understand price dynamics and potential areas of interest. This model serves as a valuable tool for strategic decision-making in financial markets.